# 秩奇异值与条件数

## Goal
手算并重构2×3矩阵，再区分矩阵近似、任务信息与逆问题敏感性。主例奇异值4、2，秩一误差2；百万分之一右端扰动在特定弱方向造成100%解变化。所有数据为本讲原创合成数值，无网络和GPU。

## Setup
在013目录启动并重启内核后运行全部。核心只需Python与NumPy。

### Key Assumptions
有限float64实数；行样本xW=b；条件数推导只针对可逆方阵与非零右端，固定W，只扰动b。不训练网络，不预设导数。

In [1]:
from pathlib import Path
import json
import numpy as np
import experiment as lab
config = json.loads((lab.ROOT / "data/config.json").read_text())
W = lab.matrix(config["matrix"])
print("W shape:", W.shape, "NumPy:", np.__version__)
print(W)

W shape: (2, 3) NumPy: 2.3.5
[[3. 1. 0.]
 [1. 3. 0.]]


## Steps
### 1 核对手算平方矩阵
WWᵀ应为[[10,6],[6,10]]；特征值16、4的平方根才是奇异值。这里用平方矩阵辅助理解，不把它当作数值SVD算法。

In [2]:
U_hand = np.array([[1.,1.],[1.,-1.]]) / np.sqrt(2)
print(W @ W.T)
print((W @ W.T) @ U_hand)
assert np.allclose((W @ W.T) @ U_hand, U_hand @ np.diag([16.,4.]))

[[10.  6.]
 [ 6. 10.]]
[[11.3137085   2.82842712]
 [11.3137085  -2.82842712]]


### 2 约简SVD
Vt已是转置因子。向量符号可以同时翻转；检查乘积与正交性，而不是逐项强求固定符号。

In [3]:
U, s, Vt = lab.reduced_svd(W)
print("shapes", U.shape, s.shape, Vt.shape)
print("singular values", s)
print("reconstruction", (U*s) @ Vt)
assert np.allclose(U.T @ U, np.eye(2))
assert np.allclose(Vt @ Vt.T, np.eye(2))
assert np.allclose((U*s) @ Vt, W)

shapes (2, 2) (2,) (2, 3)
singular values [4. 2.]
reconstruction [[3. 1. 0.]
 [1. 3. 0.]]


### 3 截断所有秩预算
比较观察误差与尾部平方和开根。k是方向预算，矩阵输出形状仍为2×3。

In [4]:
for k in range(3):
    approximation = lab.rank_k(W, k)
    observed = lab.frobenius(W - approximation)
    tail = float(np.sqrt(np.sum(s[k:]**2)))
    print(k, "observed", observed, "tail", tail)
    assert np.isclose(observed, tail, atol=1e-12)
print("rank one", lab.rank_k(W, 1))
print("relative Frobenius error", 2/np.sqrt(20))

0 observed 4.47213595499958 tail 4.47213595499958
1 observed 2.0 tail 1.9999999999999996
2 observed 6.661338147750939e-16 tail 0.0
rank one [[2. 2. 0.]
 [2. 2. 0.]]
relative Frobenius error 0.4472135954999579


### 4 最坏方向与安全方向
两个单位输入在同一截断矩阵下的误差不同。

In [5]:
error_matrix = W - lab.rank_k(W, 1)
for x in [np.array([[1.,1.]])/np.sqrt(2), np.array([[1.,-1.]])/np.sqrt(2)]:
    print("input", x, "output error norm", lab.frobenius(x @ error_matrix))

input [[0.70710678 0.70710678]] output error norm 0.0
input [[ 0.70710678 -0.70710678]] output error norm 1.9999999999999998


### 5 小能量方向中的全部标签
两类标签取决于第二坐标符号。截断后两个输出完全相同，不能通过更复杂的后续确定性分类器找回。

In [6]:
H = np.diag([10., .1])
X = np.array([[1.,1.],[1.,-1.]])
print("full", X @ H)
print("rank one", X @ lab.rank_k(H, 1))
print("relative error", lab.frobenius(H-lab.rank_k(H,1))/lab.frobenius(H))
assert np.array_equal((X @ lab.rank_k(H,1))[0], (X @ lab.rank_k(H,1))[1])

full [[10.   0.1]
 [10.  -0.1]]
rank one [[10.  0.]
 [10.  0.]]
relative error 0.009999500037496875


### 6 沿弱方向扰动右端
相同δ，ε越小，解变化越大。condition2是最坏情况界；这里特意构造达到它的方向。

In [7]:
for epsilon in config["epsilons"]:
    result = lab.perturbation_case(epsilon, config["delta"])
    print(epsilon, result["relative_b"], result["relative_x"], result["amplification"])
print("last perturbed x", result["perturbed_x"])

1.0 1e-06 1e-06 1.0
0.01 1e-06 9.999999999999999e-05 100.0
0.0001 1e-06 0.009999999999999998 9999.999999999998
1e-06 1e-06 1.0 1000000.0
last perturbed x [[1.0, 1.0]]


### 7 安全方向与残差
先算强方向扰动，再区分候选解对原右端和新右端的残差。

In [8]:
ill = np.diag([1.,1e-6])
b = np.array([[1.,0.]])
print("strong-direction solution", lab.solve_row(ill, [[1.+1e-6,0.]]))
x_hat = np.array([[1.,1.]])
print("residual relative to original b", x_hat @ ill - b)
print("residual relative to perturbed b", x_hat @ ill - np.array([[1.,1e-6]]))
print("relative solution error", lab.frobenius(x_hat - np.array([[1.,0.]])))

strong-direction solution [[1.000001 0.      ]]
residual relative to original b [[0.e+00 1.e-06]]
residual relative to perturbed b [[0. 0.]]
relative solution error 1.0


### 8 公开数值秩阈值
数学秩为2。绝对阈值固定而整体尺度改变，会改变数值秩判定。

In [9]:
small = np.diag([1.,1e-8])
for tolerance in [1e-10,1e-6]:
    print(tolerance, lab.numerical_rank(small,tolerance), lab.numerical_rank(1e6*small,tolerance))
print("strict threshold boundary", lab.numerical_rank(np.diag([1.,.5]),.5))

1e-10 2 2
1e-06 1 2
strict threshold boundary 1


### 9 低秩增量不等于完整权重低秩
这里只验证因子乘积，不训练LoRA或大型模型。

In [10]:
W0 = np.eye(2)
A = np.array([[1.],[0.]])
B = np.array([[1.,0.]])
print("increment", A @ B)
print("complete weight", W0 + A @ B)
assert lab.numerical_rank(A @ B, 1e-12) == 1
assert lab.numerical_rank(W0 + A @ B, 1e-12) == 2

increment [[1. 0.]
 [0. 0.]]
complete weight [[2. 0.]
 [0. 1.]]


## Checks
### 10 主动检查失败输入
不接受布尔秩预算。程序应抛出清楚的异常，不能无声接受True为1。

In [11]:
try:
    lab.rank_k(W, True)
except TypeError as error:
    print(type(error).__name__, str(error))
else:
    raise AssertionError("Boolean k must be rejected")

TypeError k must be an integer, not Boolean


### 11 独立参考与全部测试
包含100个80位Decimal解析SVD参考、28个Fraction行方程、边界与溢出失败测试。

In [12]:
import test_experiment
test_report = test_experiment.main()
assert test_report["status"] == "passed"

(2, 2) (2,) (2, 3)
[[2. 2. 0.]
 [2. 2. 0.]] 2.0
[[2. 2.]]
[[1.0, 1.0]]
0 4.47213595499958
1 2.0
2 6.661338147750939e-16
TypeError k must be an integer, not Boolean
{
  "status": "passed",
  "groups": 8,
  "test_groups": [
    "fixed rectangular hand SVD, shapes, orthogonality, rank truncations",
    "100 independent 80-digit Decimal 2x2 singular-value and tail oracles",
    "28 independent Fraction nonsymmetric row-system oracles",
    "exact rational perturbation amplification and safe direction",
    "rank tolerance boundary, zero, singular, negative eigenvalue and repeated spectrum",
    "explicit type/shape/range/overflow rejections and input immutability",
    "repeat byte-identical outputs, information loss, invalid config fails before output",
    "all Python document blocks executed in order"
  ],
  "decimal_fixtures": 100,
  "fraction_solve_fixtures": 28,
  "rejected_cases": 41,
  "document_snippets": 5
}


### 12 保存可复查结果
核心数值函数与脚本共用。输出目录与脚本分开，避免互相覆盖。

In [13]:
report = lab.run(Path("notebook_outputs"))
print("saved files", sorted(p.name for p in Path("notebook_outputs").iterdir()))
print("singular values", report["singular_values"])

saved files ['environment.json', 'perturbations.csv', 'rank_errors.csv', 'results.json']
singular values [4.0, 1.9999999999999996]


## Next Steps
用lab.pdf的A至H清单写解释，尤其区分数学秩、数值秩、残差和解误差。当前数据只演示机制。已在新进程真实InProcessKernel顺序执行；未测试浏览器Jupyter界面、跨进程socket传输或新conda安装。

来源见lecture与source_checks.md：NumPy 2.3官方SVD/solve/matrix_rank文档、LAPACK Users’ Guide、MIT Strang讲授材料和LoRA原论文。